In [1]:
## KATA 7.1

In [6]:
# Prompt-
'''
I am setting up a local data analysis environment using DuckDB and
Python in a {{NOTEBOOK_ENV: Jupyter / Google Colab}} notebook.

Generate a complete notebook cell that:
1. Installs duckdb and pandas using pip (with --quiet flag)
2. Imports duckdb, pandas as pd, os, random, numpy as np, datetime
3. Creates an in-memory DuckDB connection
4. Creates a test table hello_world (id INTEGER, message VARCHAR,
   created_at TIMESTAMP)
5. Inserts 3 rows of sample data
6. Runs a SELECT query and prints the results
7. Prints "Environment ready ✓" at the end
Make the code compatible with both Google Colab and local Jupyter.

'''

'\nI am setting up a local data analysis environment using DuckDB and\nPython in a {{NOTEBOOK_ENV: Jupyter / Google Colab}} notebook.\n\nGenerate a complete notebook cell that:\n1. Installs duckdb and pandas using pip (with --quiet flag)\n2. Imports duckdb, pandas as pd, os, random, numpy as np, datetime\n3. Creates an in-memory DuckDB connection\n4. Creates a test table hello_world (id INTEGER, message VARCHAR,\n   created_at TIMESTAMP)\n5. Inserts 3 rows of sample data\n6. Runs a SELECT query and prints the results\n7. Prints "Environment ready ✓" at the end\nMake the code compatible with both Google Colab and local Jupyter.\n\n'

In [5]:
# # DuckDB + Python Environment Setup — Line-by-Line Explanation

# ## Setup: install dependencies
# - `!pip install --quiet duckdb pandas`
#   Uses the notebook shell magic (`!`) to run pip. Installs **duckdb** (the database engine) and **pandas** (dataframes). `--quiet` suppresses the verbose install log.

# ## Imports
# - `import duckdb` — the in-process SQL analytics engine.
# - `import pandas as pd` — dataframes; also lets DuckDB hand back results as `pd.DataFrame`.
# - `import os` — operating-system helpers (paths, env vars); available for later cells.
# - `import random` — random number generation; available for later cells.
# - `import numpy as np` — numerical arrays; available for later cells.
# - `import datetime` — date/time objects, used here to build the `created_at` timestamps.

# ## Create an in-memory DuckDB connection
# - `con = duckdb.connect(database=":memory:")`
#   Opens a DuckDB connection that lives entirely in RAM (`:memory:`). Nothing is written to disk, and the data disappears when the kernel restarts. `con` is the handle you reuse for every query.

# ## Create the test table
# - `con.execute("""CREATE TABLE hello_world (...)""")`
#   Runs a SQL `CREATE TABLE` statement defining three columns: `id` (INTEGER), `message` (VARCHAR / text), and `created_at` (TIMESTAMP).

# ## Insert 3 rows of sample data
# - `now = datetime.datetime.now()`
#   Captures the current date and time as a starting reference point.
# - `rows = [ (1, ...), (2, ...), (3, ...) ]`
#   A Python list of 3 tuples, each matching the table's `(id, message, created_at)` columns. `datetime.timedelta(minutes=n)` offsets each row's timestamp so they differ.
# - `con.executemany("INSERT ... VALUES (?, ?, ?)", rows)`
#   Inserts all rows in one call. The `?` are **parameter placeholders** — DuckDB safely substitutes each tuple's values, avoiding SQL-injection and quoting issues.

# ## Run a SELECT query and print the results
# - `result_df = con.execute("SELECT * FROM hello_world ORDER BY id").fetchdf()`
#   Selects all rows, sorted by `id`. `.fetchdf()` returns the result as a pandas DataFrame.
# - `print(result_df)`
#   Prints the DataFrame as a formatted table.

# ## Done
# - `print("Environment ready ✓")`
#  Confirms the cell ran end-to-end without errors.

In [2]:
# ── Setup: install dependencies ───────────────────────────────
!pip install --quiet duckdb pandas

# ── Imports ───────────────────────────────────────────────────
import duckdb
import pandas as pd
import os
import random
import numpy as np
import datetime

# ── Create an in-memory DuckDB connection ─────────────────────
con = duckdb.connect(database=":memory:")

# ── Create the test table ─────────────────────────────────────
con.execute("""
    CREATE TABLE hello_world (
        id         INTEGER,
        message    VARCHAR,
        created_at TIMESTAMP
    )
""")

# ── Insert 3 rows of sample data ──────────────────────────────
now = datetime.datetime.now()
rows = [
    (1, "Hello, DuckDB!",   now),
    (2, "Local analytics",  now + datetime.timedelta(minutes=1)),
    (3, "Environment test", now + datetime.timedelta(minutes=2)),
]
con.executemany(
    "INSERT INTO hello_world (id, message, created_at) VALUES (?, ?, ?)",
    rows,
)

# ── Run a SELECT query and print the results ──────────────────
result_df = con.execute("SELECT * FROM hello_world ORDER BY id").fetchdf()
print(result_df)

# ── Done ──────────────────────────────────────────────────────
print("Environment ready ✓")


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


   id           message                 created_at
0   1    Hello, DuckDB! 2026-07-01 19:50:28.969370
1   2   Local analytics 2026-07-01 19:51:28.969370
2   3  Environment test 2026-07-01 19:52:28.969370
Environment ready ✓


## KATA 7.2

In [9]:
import random
import numpy as np
import pandas as pd
import os
from datetime import datetime, timedelta

# ============================================================
# Set seeds for full reproducibility
# ============================================================
random.seed(42)
np.random.seed(42)

# ============================================================
# Configuration constants
# ============================================================
NUM_ROWS = 500
DUPLICATE_RATE = 0.03        # 3% duplicate order_ids
NULL_AMOUNT_RATE = 0.05      # 5% null amounts
NEGATIVE_AMOUNT_RATE = 0.02  # 2% negative amounts (returns)

REGIONS = ["North", "South", "East", "West"]
CATEGORIES = ["Electronics", "Clothing", "Food", "Home", "Sports"]
STATUSES = ["completed", "returned", "pending"]
STATUS_WEIGHTS = [0.80, 0.15, 0.05]

DATE_START = datetime(2024, 1, 1)
DATE_END = datetime(2024, 12, 31)
DATE_RANGE_DAYS = (DATE_END - DATE_START).days

# ============================================================
# Helper: generate a random date within 2024
# ============================================================
def random_date():
    offset = random.randint(0, DATE_RANGE_DAYS)
    return DATE_START + timedelta(days=offset)

# ============================================================
# Helper: format a date in one of three random formats
#   "2024-01-15"   — ISO / YYYY-MM-DD
#   "15/01/2024"   — DD/MM/YYYY
#   "Jan 15 2024"  — Mon DD YYYY
# ============================================================
def format_date(dt):
    fmt = random.choice(["iso", "dmy", "mdy_text"])
    if fmt == "iso":
        return dt.strftime("%Y-%m-%d")
    elif fmt == "dmy":
        return dt.strftime("%d/%m/%Y")
    else:
        return dt.strftime("%b %d %Y")

# ============================================================
# STEP 1: Generate unique base order_ids, then introduce dupes
# ============================================================
num_duplicates = int(NUM_ROWS * DUPLICATE_RATE)   # 15 duplicates
num_unique = NUM_ROWS - num_duplicates             # 485 unique ids

# Create 485 unique order IDs of the form "ORD-XXXXX"
unique_ids = random.sample(range(10000, 99999), num_unique)
order_ids = [f"ORD-{uid}" for uid in unique_ids]

# Pick 15 existing IDs at random and duplicate them
duplicates = random.choices(order_ids, k=num_duplicates)
order_ids.extend(duplicates)

# Shuffle so duplicates are scattered throughout the file
random.shuffle(order_ids)

# ============================================================
# STEP 2: Generate the remaining columns
# ============================================================
customer_ids = [random.randint(1000, 9999) for _ in range(NUM_ROWS)]
regions = [random.choice(REGIONS) for _ in range(NUM_ROWS)]
order_dates = [format_date(random_date()) for _ in range(NUM_ROWS)]
categories = [random.choice(CATEGORIES) for _ in range(NUM_ROWS)]
quantities = [random.randint(1, 10) for _ in range(NUM_ROWS)]
statuses = random.choices(STATUSES, weights=STATUS_WEIGHTS, k=NUM_ROWS)

# ============================================================
# STEP 3: Generate amounts with 5% nulls and 2% negatives
# ============================================================
amounts = []
for i in range(NUM_ROWS):
    roll = random.random()
    if roll < NULL_AMOUNT_RATE:
        amounts.append(np.nan)                          # 5% null
    elif roll < NULL_AMOUNT_RATE + NEGATIVE_AMOUNT_RATE:
        amounts.append(-round(random.uniform(5, 500), 2))  # 2% negative
    else:
        amounts.append(round(random.uniform(5, 500), 2))   # normal

# ============================================================
# STEP 4: Assemble into a DataFrame
# ============================================================
df = pd.DataFrame({
    "order_id":         order_ids,
    "customer_id":      customer_ids,
    "region":           regions,
    "order_date":       order_dates,
    "product_category": categories,
    "amount":           amounts,
    "quantity":          quantities,
    "status":           statuses,
})

# ============================================================
# STEP 5: Save to bronze/transactions_raw.csv
# ============================================================
output_dir = "bronze"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, "transactions_raw.csv")
df.to_csv(output_path, index=False)

# ============================================================
# STEP 6: Print summary statistics
# ============================================================
total_rows = len(df)
null_count = df["amount"].isna().sum()
duplicate_count = df["order_id"].duplicated(keep=False).sum()

# Detect unique date format patterns in the data
def detect_format(date_str):
    if "-" in date_str and date_str[:4].isdigit():
        return "YYYY-MM-DD"
    elif "/" in date_str:
        return "DD/MM/YYYY"
    else:
        return "Mon DD YYYY"

unique_formats = sorted(set(detect_format(d) for d in df["order_date"]))

print(f"Total rows          : {total_rows}")
print(f"Null amount count   : {null_count}")
print(f"Duplicate order_ids : {duplicate_count} rows ({df['order_id'].duplicated().sum()} non-first occurrences)")
print(f"Unique date formats : {len(unique_formats)} — {unique_formats}")
print(f"\nSaved to: {output_path}")
print(f"\n--- First 5 rows ---")
print(df.head().to_string(index=False))


Total rows          : 500
Null amount count   : 24
Duplicate order_ids : 28 rows (15 non-first occurrences)
Unique date formats : 3 — ['DD/MM/YYYY', 'Mon DD YYYY', 'YYYY-MM-DD']

Saved to: bronze\transactions_raw.csv

--- First 5 rows ---
 order_id  customer_id region  order_date product_category  amount  quantity    status
ORD-69638         8679   West  25/07/2024      Electronics  456.19         4 completed
ORD-46509         5982  South  24/01/2024             Home  325.58         9  returned
ORD-54313         7594  South Jan 27 2024             Home  275.57         4  returned
ORD-30676         5460   West  2024-10-14             Food  389.19         7 completed
ORD-33509         9199  South Jul 04 2024      Electronics  150.62         8 completed


##
Write DuckDB SQL to profile transactions_raw.csv: row count, null count per column, duplicate order_id count, min/max amount, distinct status values

also store the output of queries in artefacts/700-wide/bronze-profile.md

In [11]:
import duckdb
import os

# ============================================================
# Connect to DuckDB (in-memory) and point it at the CSV
# ============================================================
con = duckdb.connect(database=":memory:")

CSV_PATH = "bronze/transactions_raw.csv"

# ============================================================
# Create output directory
# ============================================================
os.makedirs("artefacts", exist_ok=True)
OUTPUT_PATH = "artefacts/bronze-profile.md"

# We'll collect each section of markdown as we go
sections = []

# ============================================================
# 1. ROW COUNT
# ============================================================
row_count = con.execute(f"""
    SELECT COUNT(*) AS total_rows
    FROM   read_csv_auto('{CSV_PATH}')
""").fetchone()[0]

sections.append(f"""\
# Bronze Layer — Data Profile
## `transactions_raw.csv`

---

### 1. Row Count

| Metric     | Value |
|:-----------|------:|
| Total Rows | {row_count:,} |
""")

print(f"Total rows: {row_count}")

# ============================================================
# 2. NULL COUNT PER COLUMN
# ============================================================
null_query = f"""
    SELECT
        SUM(CASE WHEN order_id         IS NULL THEN 1 ELSE 0 END) AS order_id_nulls,
        SUM(CASE WHEN customer_id      IS NULL THEN 1 ELSE 0 END) AS customer_id_nulls,
        SUM(CASE WHEN region           IS NULL THEN 1 ELSE 0 END) AS region_nulls,
        SUM(CASE WHEN order_date       IS NULL THEN 1 ELSE 0 END) AS order_date_nulls,
        SUM(CASE WHEN product_category IS NULL THEN 1 ELSE 0 END) AS product_category_nulls,
        SUM(CASE WHEN amount           IS NULL THEN 1 ELSE 0 END) AS amount_nulls,
        SUM(CASE WHEN quantity         IS NULL THEN 1 ELSE 0 END) AS quantity_nulls,
        SUM(CASE WHEN status           IS NULL THEN 1 ELSE 0 END) AS status_nulls
    FROM read_csv_auto('{CSV_PATH}')
"""
null_result = con.execute(null_query).fetchone()
null_columns = [
    "order_id", "customer_id", "region", "order_date",
    "product_category", "amount", "quantity", "status"
]

null_table_rows = ""
for col, val in zip(null_columns, null_result):
    pct = (val / row_count) * 100
    flag = " [!]" if val > 0 else ""
    null_table_rows += f"| {col:<20} | {val:>6} | {pct:>8.1f}% |{flag}\n"

sections.append(f"""\
### 2. Null Count per Column

| Column               | Nulls  |   % Null |
|:---------------------|-------:|---------:|
{null_table_rows}""")

print("\nNull counts per column:")
for col, val in zip(null_columns, null_result):
    print(f"  {col:20s}: {val}")

# ============================================================
# 3. DUPLICATE ORDER_ID COUNT
# ============================================================
dup_query = f"""
    WITH id_counts AS (
        SELECT order_id,
               COUNT(*) AS cnt
        FROM   read_csv_auto('{CSV_PATH}')
        GROUP  BY order_id
        HAVING COUNT(*) > 1
    )
    SELECT
        COUNT(*)        AS duplicate_groups,
        SUM(cnt)        AS total_rows_involved
    FROM id_counts
"""
dup_groups, dup_rows = con.execute(dup_query).fetchone()

sections.append(f"""\
### 3. Duplicate `order_id` Analysis

| Metric                          | Value |
|:--------------------------------|------:|
| Distinct order_ids with dupes   | {dup_groups:>5} |
| Total rows involved in dupes    | {dup_rows:>5} |
| Duplication rate                | {(dup_rows / row_count) * 100:.1f}% |
""")

print(f"\nDuplicate order_ids: {dup_groups} groups, {dup_rows} rows involved")

# ============================================================
# 4. AMOUNT — MIN / MAX / STATS
# ============================================================
amount_query = f"""
    SELECT
        MIN(amount)                          AS min_amount,
        MAX(amount)                          AS max_amount,
        ROUND(AVG(amount), 2)                AS avg_amount,
        ROUND(MEDIAN(amount), 2)             AS median_amount,
        SUM(CASE WHEN amount < 0 THEN 1
                  ELSE 0 END)                AS negative_count
    FROM read_csv_auto('{CSV_PATH}')
    WHERE amount IS NOT NULL
"""
min_amt, max_amt, avg_amt, med_amt, neg_count = con.execute(amount_query).fetchone()

sections.append(f"""\
### 4. Amount Range & Statistics

| Metric          |    Value |
|:----------------|--------:|
| Min amount      | {min_amt:>8.2f} |
| Max amount      | {max_amt:>8.2f} |
| Avg amount      | {avg_amt:>8.2f} |
| Median amount   | {med_amt:>8.2f} |
| Negative (returns) | {neg_count:>5} |
""")

print(f"\nAmount -- min: {min_amt}, max: {max_amt}, avg: {avg_amt}, median: {med_amt}, negatives: {neg_count}")

# ============================================================
# 5. DISTINCT STATUS VALUES
# ============================================================
status_query = f"""
    SELECT   status,
             COUNT(*)                       AS row_count,
             ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS pct
    FROM     read_csv_auto('{CSV_PATH}')
    GROUP BY status
    ORDER BY row_count DESC
"""
status_rows = con.execute(status_query).fetchall()

status_table = ""
for s, cnt, pct in status_rows:
    status_table += f"| {s:<12} | {cnt:>5} | {pct:>6.1f}% |\n"

sections.append(f"""\
### 5. Distinct Status Values

| Status       | Count |     %  |
|:-------------|------:|-------:|
{status_table}""")

print("\nStatus distribution:")
for s, cnt, pct in status_rows:
    print(f"  {s:12s}: {cnt:>4} ({pct:.1f}%)")

# ============================================================
# 6. BONUS — DATE FORMAT MIX (detected via SQL pattern matching)
# ============================================================
# NOTE: Using raw strings (r"...") for regex patterns to avoid
# Python SyntaxWarning about invalid escape sequences.
date_fmt_query = (
    "SELECT "
    "  CASE "
    r"    WHEN order_date ~ '^\d{4}-\d{2}-\d{2}$'         THEN 'YYYY-MM-DD' "
    r"    WHEN order_date ~ '^\d{2}/\d{2}/\d{4}$'         THEN 'DD/MM/YYYY' "
    r"    WHEN order_date ~ '^[A-Z][a-z]{2} \d{2} \d{4}$' THEN 'Mon DD YYYY' "
    "    ELSE 'Unknown' "
    "  END AS date_format, "
    "  COUNT(*) AS row_count, "
    "  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS pct "
    f"FROM read_csv_auto('{CSV_PATH}', types={{'order_date': 'VARCHAR'}}) "
    "GROUP BY date_format "
    "ORDER BY row_count DESC"
)
date_rows = con.execute(date_fmt_query).fetchall()

date_table = ""
for fmt, cnt, pct in date_rows:
    date_table += f"| `{fmt:<12}` | {cnt:>5} | {pct:>6.1f}% |\n"

sections.append(f"""\
### 6. Date Format Distribution

| Format         | Count |     %  |
|:---------------|------:|-------:|
{date_table}
---

*Profile generated by DuckDB {duckdb.__version__} -- in-memory, zero-copy CSV scan.*
""")

print("\nDate format distribution:")
for fmt, cnt, pct in date_rows:
    print(f"  {fmt:12s}: {cnt:>4} ({pct:.1f}%)")

# ============================================================
# WRITE THE MARKDOWN FILE (explicit UTF-8 for Windows compat)
# ============================================================
md_content = "\n".join(sections)

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    f.write(md_content)

print(f"\n[OK] Profile saved to {OUTPUT_PATH}")


Total rows: 500

Null counts per column:
  order_id            : 0
  customer_id         : 0
  region              : 0
  order_date          : 0
  product_category    : 0
  amount              : 24
  quantity            : 0
  status              : 0

Duplicate order_ids: 13 groups, 28 rows involved

Amount -- min: -466.53, max: 499.22, avg: 239.97, median: 243.77, negatives: 8

Status distribution:
  completed   :  399 (79.8%)
  returned    :   78 (15.6%)
  pending     :   23 (4.6%)

Date format distribution:
  Mon DD YYYY :  182 (36.4%)
  DD/MM/YYYY  :  177 (35.4%)
  YYYY-MM-DD  :  141 (28.2%)

[OK] Profile saved to artefacts/bronze-profile.md


# KATA 7.3

In [12]:
'''
I have a DuckDB table from bronze/transactions_raw.csv with:
- ~{{NULL_COUNT}} rows with null amount
- ~{{DUP_COUNT}} rows with duplicate order_id (keep highest customer_id)
- Mixed date formats in order_date: "2024-01-15", "15/01/2024",
  "Jan 15 2024" — standardize all to DATE
- ~10 rows with negative amount (legitimate returns, do NOT remove)

Write DuckDB SQL that:
1. Reads bronze/transactions_raw.csv
2. Removes rows where amount IS NULL
3. Standardizes order_date to DATE using TRY_STRPTIME for all three formats
4. Deduplicates by order_id, keeping the highest customer_id
5. Writes to silver/transactions_clean.parquet (COPY ... TO FORMAT PARQUET)
6. Runs a verification query printing: silver row count, null amount
   count, duplicate order_id count
Do NOT remove negative amount rows — those are valid returns.
'''

'\nI have a DuckDB table from bronze/transactions_raw.csv with:\n- ~{{NULL_COUNT}} rows with null amount\n- ~{{DUP_COUNT}} rows with duplicate order_id (keep highest customer_id)\n- Mixed date formats in order_date: "2024-01-15", "15/01/2024",\n  "Jan 15 2024" — standardize all to DATE\n- ~10 rows with negative amount (legitimate returns, do NOT remove)\n\nWrite DuckDB SQL that:\n1. Reads bronze/transactions_raw.csv\n2. Removes rows where amount IS NULL\n3. Standardizes order_date to DATE using TRY_STRPTIME for all three formats\n4. Deduplicates by order_id, keeping the highest customer_id\n5. Writes to silver/transactions_clean.parquet (COPY ... TO FORMAT PARQUET)\n6. Runs a verification query printing: silver row count, null amount\n   count, duplicate order_id count\nDo NOT remove negative amount rows — those are valid returns.\n'

In [13]:
import duckdb
import os

# ============================================================
# Connect to DuckDB (in-memory)
# ============================================================
con = duckdb.connect(database=":memory:")

BRONZE_CSV    = "bronze/transactions_raw.csv"
SILVER_DIR    = "silver"
SILVER_PARQUET = os.path.join(SILVER_DIR, "transactions_clean.parquet")

os.makedirs(SILVER_DIR, exist_ok=True)

# ============================================================
# STEP 0: Quick bronze-layer snapshot (before cleaning)
# ============================================================
before = con.execute(f"""
    SELECT
        COUNT(*)                                          AS total_rows,
        SUM(CASE WHEN amount IS NULL THEN 1 ELSE 0 END)  AS null_amounts,
        SUM(CASE WHEN amount < 0    THEN 1 ELSE 0 END)   AS negative_amounts
    FROM read_csv_auto('{BRONZE_CSV}')
""").fetchone()

print("=== BRONZE (before cleaning) ===")
print(f"  Total rows       : {before[0]}")
print(f"  Null amounts     : {before[1]}")
print(f"  Negative amounts : {before[2]} (returns -- will be kept)")

# ============================================================
# STEP 1-4: Read, clean, standardize dates, deduplicate
#
#   Layer 1 (raw_with_dates):
#       - Read CSV with order_date forced to VARCHAR
#       - Drop rows where amount IS NULL
#       - Standardize order_date to DATE using COALESCE +
#         TRY_STRPTIME across all three known formats
#
#   Layer 2 (deduped):
#       - QUALIFY with ROW_NUMBER to keep only the row with
#         the highest customer_id per order_id
#       - Ties broken by ROWID (arbitrary but deterministic)
#
# ============================================================
clean_query = f"""
    WITH raw_with_dates AS (
        SELECT
            order_id,
            customer_id,
            region,
            -- Try each date format in turn; first successful parse wins
            COALESCE(
                TRY_STRPTIME(order_date, '%Y-%m-%d'),   -- 2024-01-15
                TRY_STRPTIME(order_date, '%d/%m/%Y'),   -- 15/01/2024
                TRY_STRPTIME(order_date, '%b %d %Y')    -- Jan 15 2024
            )::DATE AS order_date,
            product_category,
            amount,
            quantity,
            status
        FROM read_csv_auto('{BRONZE_CSV}', types={{'order_date': 'VARCHAR'}})
        WHERE amount IS NOT NULL
    ),

    deduped AS (
        SELECT *
        FROM   raw_with_dates
        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY order_id
            ORDER BY     customer_id DESC
        ) = 1
    )

    SELECT * FROM deduped
    ORDER BY order_date, order_id
"""

# Preview the cleaned result before writing
preview_df = con.execute(clean_query).fetchdf()
print(f"\n=== CLEANING RESULT ===")
print(f"  Rows after cleaning : {len(preview_df)}")
print(f"\n  First 5 rows:")
print(preview_df.head().to_string(index=False))

# ============================================================
# STEP 5: Write to silver/transactions_clean.parquet
# ============================================================
con.execute(f"""
    COPY (
        {clean_query}
    ) TO '{SILVER_PARQUET}' (FORMAT PARQUET)
""")

print(f"\n  Parquet written to  : {SILVER_PARQUET}")

# ============================================================
# STEP 6: Verification query on the silver parquet file
# ============================================================
verify = con.execute(f"""
    SELECT
        COUNT(*)                                          AS silver_row_count,
        SUM(CASE WHEN amount IS NULL THEN 1 ELSE 0 END)  AS null_amount_count,
        SUM(CASE WHEN amount < 0    THEN 1 ELSE 0 END)   AS negative_amount_count
    FROM '{SILVER_PARQUET}'
""").fetchone()

dup_check = con.execute(f"""
    SELECT COUNT(*) AS duplicate_order_ids
    FROM (
        SELECT   order_id
        FROM     '{SILVER_PARQUET}'
        GROUP BY order_id
        HAVING   COUNT(*) > 1
    )
""").fetchone()[0]

date_type_check = con.execute(f"""
    SELECT typeof(order_date) AS date_type
    FROM   '{SILVER_PARQUET}'
    LIMIT  1
""").fetchone()[0]

print(f"\n=== SILVER VERIFICATION ===")
print(f"  Row count          : {verify[0]}")
print(f"  Null amounts       : {verify[1]}")
print(f"  Negative amounts   : {verify[2]} (returns -- correctly kept)")
print(f"  Duplicate order_ids: {dup_check}")
print(f"  order_date type    : {date_type_check}")

# ============================================================
# Final summary
# ============================================================
rows_removed = before[0] - verify[0]
print(f"\n=== SUMMARY ===")
print(f"  Bronze rows        : {before[0]}")
print(f"  Silver rows        : {verify[0]}")
print(f"  Rows removed       : {rows_removed}")
print(f"    - Null amounts   : {before[1]}")
print(f"    - Duplicates     : {rows_removed - before[1]}")
print(f"  Negative amounts   : {verify[2]} (kept as valid returns)")
print(f"\n[OK] Bronze -> Silver pipeline complete.")

=== BRONZE (before cleaning) ===
  Total rows       : 500
  Null amounts     : 24
  Negative amounts : 8 (returns -- will be kept)

=== CLEANING RESULT ===
  Rows after cleaning : 461

  First 5 rows:
 order_id  customer_id region order_date product_category  amount  quantity    status
ORD-84364         9022   East 2024-01-03             Home  271.62         2 completed
ORD-82512         7730  North 2024-01-05      Electronics  308.95         6 completed
ORD-17492         7105   East 2024-01-07             Home  311.23         2  returned
ORD-15778         6383  North 2024-01-09      Electronics  482.94         6 completed
ORD-89507         5542  South 2024-01-09             Home  332.70         7 completed

  Parquet written to  : silver\transactions_clean.parquet

=== SILVER VERIFICATION ===
  Row count          : 461
  Null amounts       : 0
  Negative amounts   : 8 (returns -- correctly kept)
  Duplicate order_ids: 0
  order_date type    : DATE

=== SUMMARY ===
  Bronze rows       

# KATA 7.4

In [14]:
'''I have cleaned retail transactions at silver/transactions_clean.parquet
(order_id, customer_id, region, order_date DATE, product_category, amount FLOAT
[negative for returns], quantity, status [completed/returned/pending]).

Write DuckDB SQL for two gold tables:

Table 1: gold/daily_sales_by_category
- Grain: one row per (order_date, region, product_category) — daily sales
  by region (the spec grain), with product_category as a drill dimension
- total_revenue (SUM amount WHERE status='completed'),
  order_count (distinct order_ids WHERE status='completed')
- Write to gold/daily_sales_by_category.parquet

Table 2: gold/returns_rate
- Grain: one row per order_date
- total_orders (completed + returned, exclude pending),
  returned_orders (status='returned'),
  returns_rate_pct (returned_orders / total_orders * 100, 2 decimals)
- Write to gold/returns_rate.parquet

After writing, verify: exactly one row per (order_date, region, product_category);
returns_rate_pct between 0 and 100; print row counts for both tables.'''

"I have cleaned retail transactions at silver/transactions_clean.parquet\n(order_id, customer_id, region, order_date DATE, product_category, amount FLOAT\n[negative for returns], quantity, status [completed/returned/pending]).\n\nWrite DuckDB SQL for two gold tables:\n\nTable 1: gold/daily_sales_by_category\n- Grain: one row per (order_date, region, product_category) — daily sales\n  by region (the spec grain), with product_category as a drill dimension\n- total_revenue (SUM amount WHERE status='completed'),\n  order_count (distinct order_ids WHERE status='completed')\n- Write to gold/daily_sales_by_category.parquet\n\nTable 2: gold/returns_rate\n- Grain: one row per order_date\n- total_orders (completed + returned, exclude pending),\n  returned_orders (status='returned'),\n  returns_rate_pct (returned_orders / total_orders * 100, 2 decimals)\n- Write to gold/returns_rate.parquet\n\nAfter writing, verify: exactly one row per (order_date, region, product_category);\nreturns_rate_pct bet

In [15]:
import duckdb
import os

# ============================================================
# Connect to DuckDB (in-memory)
# ============================================================
con = duckdb.connect(database=":memory:")

SILVER_PARQUET = "silver/transactions_clean.parquet"
GOLD_DIR       = "gold"
GOLD_SALES     = os.path.join(GOLD_DIR, "daily_sales_by_category.parquet")
GOLD_RETURNS   = os.path.join(GOLD_DIR, "returns_rate.parquet")

os.makedirs(GOLD_DIR, exist_ok=True)

# ============================================================
# Quick look at the silver source
# ============================================================
src = con.execute(f"""
    SELECT
        COUNT(*)                                            AS total_rows,
        COUNT(DISTINCT order_id)                            AS unique_orders,
        MIN(order_date)                                     AS min_date,
        MAX(order_date)                                     AS max_date,
        COUNT(DISTINCT region)                              AS regions,
        COUNT(DISTINCT product_category)                    AS categories
    FROM '{SILVER_PARQUET}'
""").fetchone()

print("=== SILVER SOURCE ===")
print(f"  Rows           : {src[0]}")
print(f"  Unique orders  : {src[1]}")
print(f"  Date range     : {src[2]} to {src[3]}")
print(f"  Regions        : {src[4]}")
print(f"  Categories     : {src[5]}")

# ============================================================
# GOLD TABLE 1: daily_sales_by_category
#
# Grain : one row per (order_date, region, product_category)
# Scope : only completed orders contribute to revenue/count
#
# Logic :
#   - Filter to status = 'completed'
#   - GROUP BY the three grain columns
#   - SUM(amount) as total_revenue
#   - COUNT(DISTINCT order_id) as order_count
#     (distinct because the same order_id should not be
#      double-counted if it somehow appears twice)
# ============================================================
sales_query = f"""
    SELECT
        order_date,
        region,
        product_category,
        SUM(amount)                AS total_revenue,
        COUNT(DISTINCT order_id)   AS order_count
    FROM   '{SILVER_PARQUET}'
    WHERE  status = 'completed'
    GROUP  BY order_date, region, product_category
    ORDER  BY order_date, region, product_category
"""

con.execute(f"COPY ({sales_query}) TO '{GOLD_SALES}' (FORMAT PARQUET)")
print(f"\n[OK] Written: {GOLD_SALES}")

# ============================================================
# GOLD TABLE 2: returns_rate
#
# Grain : one row per order_date
# Scope : completed + returned only (pending excluded)
#
# Logic :
#   - Filter out status = 'pending'
#   - total_orders   = COUNT(DISTINCT order_id)
#   - returned_orders = COUNT(DISTINCT order_id) WHERE returned
#   - returns_rate_pct = returned / total * 100, rounded to 2dp
#
#   NULLIF guards against division by zero on dates that
#   have zero qualifying orders (shouldn't happen, but safe).
# ============================================================
returns_query = f"""
    SELECT
        order_date,
        COUNT(DISTINCT order_id) AS total_orders,
        COUNT(DISTINCT CASE
            WHEN status = 'returned' THEN order_id
        END) AS returned_orders,
        ROUND(
            COUNT(DISTINCT CASE
                WHEN status = 'returned' THEN order_id
            END) * 100.0
            / NULLIF(COUNT(DISTINCT order_id), 0),
            2
        ) AS returns_rate_pct
    FROM   '{SILVER_PARQUET}'
    WHERE  status IN ('completed', 'returned')
    GROUP  BY order_date
    ORDER  BY order_date
"""

con.execute(f"COPY ({returns_query}) TO '{GOLD_RETURNS}' (FORMAT PARQUET)")
print(f"[OK] Written: {GOLD_RETURNS}")

# ============================================================
# VERIFICATION 1: daily_sales_by_category grain check
#   - Exactly one row per (order_date, region, product_category)
#   - No duplicate grain keys
# ============================================================
grain_check = con.execute(f"""
    SELECT COUNT(*) AS duplicates
    FROM (
        SELECT   order_date, region, product_category
        FROM     '{GOLD_SALES}'
        GROUP BY order_date, region, product_category
        HAVING   COUNT(*) > 1
    )
""").fetchone()[0]

sales_rows = con.execute(f"SELECT COUNT(*) FROM '{GOLD_SALES}'").fetchone()[0]

sales_stats = con.execute(f"""
    SELECT
        SUM(total_revenue)     AS grand_total_revenue,
        SUM(order_count)       AS grand_total_orders,
        MIN(total_revenue)     AS min_daily_revenue,
        MAX(total_revenue)     AS max_daily_revenue
    FROM '{GOLD_SALES}'
""").fetchone()

print(f"\n=== GOLD TABLE 1: daily_sales_by_category ===")
print(f"  Row count              : {sales_rows}")
print(f"  Grain duplicates       : {grain_check}  {'[OK]' if grain_check == 0 else '[FAIL]'}")
print(f"  Grand total revenue    : {sales_stats[0]:,.2f}")
print(f"  Grand total orders     : {sales_stats[1]}")
print(f"  Revenue range per cell : {sales_stats[2]:,.2f} to {sales_stats[3]:,.2f}")

# ============================================================
# VERIFICATION 2: returns_rate checks
#   - returns_rate_pct between 0 and 100
#   - Exactly one row per order_date
# ============================================================
returns_rows = con.execute(f"SELECT COUNT(*) FROM '{GOLD_RETURNS}'").fetchone()[0]

rate_bounds = con.execute(f"""
    SELECT
        MIN(returns_rate_pct)  AS min_rate,
        MAX(returns_rate_pct)  AS max_rate,
        AVG(returns_rate_pct)  AS avg_rate
    FROM '{GOLD_RETURNS}'
""").fetchone()

rate_violations = con.execute(f"""
    SELECT COUNT(*) AS out_of_range
    FROM   '{GOLD_RETURNS}'
    WHERE  returns_rate_pct < 0 OR returns_rate_pct > 100
""").fetchone()[0]

date_grain_check = con.execute(f"""
    SELECT COUNT(*) AS duplicates
    FROM (
        SELECT   order_date
        FROM     '{GOLD_RETURNS}'
        GROUP BY order_date
        HAVING   COUNT(*) > 1
    )
""").fetchone()[0]

print(f"\n=== GOLD TABLE 2: returns_rate ===")
print(f"  Row count              : {returns_rows}")
print(f"  Date grain duplicates  : {date_grain_check}  {'[OK]' if date_grain_check == 0 else '[FAIL]'}")
print(f"  Rate range             : {rate_bounds[0]}% to {rate_bounds[1]}%")
print(f"  Average return rate    : {rate_bounds[2]:.2f}%")
print(f"  Out-of-range rates     : {rate_violations}  {'[OK]' if rate_violations == 0 else '[FAIL]'}")

# ============================================================
# Preview: sample rows from each gold table
# ============================================================
print(f"\n--- daily_sales_by_category (first 5 rows) ---")
print(con.execute(f"SELECT * FROM '{GOLD_SALES}' LIMIT 5").fetchdf().to_string(index=False))

print(f"\n--- returns_rate (first 5 rows) ---")
print(con.execute(f"SELECT * FROM '{GOLD_RETURNS}' LIMIT 5").fetchdf().to_string(index=False))

# ============================================================
# Final pipeline summary
# ============================================================
print(f"\n=== PIPELINE SUMMARY ===")
print(f"  Silver rows                      : {src[0]}")
print(f"  Gold daily_sales_by_category     : {sales_rows} rows")
print(f"  Gold returns_rate                : {returns_rows} rows")
print(f"  All grain checks passed          : {'YES' if grain_check == 0 and date_grain_check == 0 else 'NO'}")
print(f"  All rate bounds valid            : {'YES' if rate_violations == 0 else 'NO'}")
print(f"\n[OK] Silver -> Gold pipeline complete.")

=== SILVER SOURCE ===
  Rows           : 461
  Unique orders  : 461
  Date range     : 2024-01-03 to 2024-12-31
  Regions        : 4
  Categories     : 5

[OK] Written: gold\daily_sales_by_category.parquet
[OK] Written: gold\returns_rate.parquet

=== GOLD TABLE 1: daily_sales_by_category ===
  Row count              : 356
  Grain duplicates       : 0  [OK]
  Grand total revenue    : 88,912.72
  Grand total orders     : 369
  Revenue range per cell : -400.64 to 799.55

=== GOLD TABLE 2: returns_rate ===
  Row count              : 250
  Date grain duplicates  : 0  [OK]
  Rate range             : 0.0% to 100.0%
  Average return rate    : 15.24%
  Out-of-range rates     : 0  [OK]

--- daily_sales_by_category (first 5 rows) ---
order_date region product_category  total_revenue  order_count
2024-01-03   East             Home         271.62            1
2024-01-05  North      Electronics         308.95            1
2024-01-09  North      Electronics         482.94            1
2024-01-09  Sou

# KATA 7.5

In [17]:
'''I have two DuckDB gold tables:
gold/daily_sales_by_category.parquet: order_date DATE,
  region VARCHAR, product_category VARCHAR, total_revenue FLOAT, order_count INTEGER
gold/returns_rate.parquet: order_date DATE, total_orders INTEGER,
  returned_orders INTEGER, returns_rate_pct FLOAT

Write Python functions using DuckDB SQL that check these rules.
Each check prints PASS or FAIL with the rule name and, on failure,
the failing row count and example values.

daily_sales_by_category:
1. No null order_date, region, or product_category
2. total_revenue > 0 for all rows
3. order_count > 0 for all rows
4. No duplicate (order_date, region, product_category) combinations
returns_rate:
5. No null order_date
6. returns_rate_pct between 0.0 and 100.0 inclusive
7. returned_orders <= total_orders
8. order_date range spans at least 30 days

Create run_all_checks() that runs all 8 and prints: X/8 checks passed.'''

'I have two DuckDB gold tables:\ngold/daily_sales_by_category.parquet: order_date DATE,\n  region VARCHAR, product_category VARCHAR, total_revenue FLOAT, order_count INTEGER\ngold/returns_rate.parquet: order_date DATE, total_orders INTEGER,\n  returned_orders INTEGER, returns_rate_pct FLOAT\n\nWrite Python functions using DuckDB SQL that check these rules.\nEach check prints PASS or FAIL with the rule name and, on failure,\nthe failing row count and example values.\n\ndaily_sales_by_category:\n1. No null order_date, region, or product_category\n2. total_revenue > 0 for all rows\n3. order_count > 0 for all rows\n4. No duplicate (order_date, region, product_category) combinations\nreturns_rate:\n5. No null order_date\n6. returns_rate_pct between 0.0 and 100.0 inclusive\n7. returned_orders <= total_orders\n8. order_date range spans at least 30 days\n\nCreate run_all_checks() that runs all 8 and prints: X/8 checks passed.'

In [18]:
import duckdb

# ============================================================
# DuckDB connection (in-memory, reads parquet directly)
# ============================================================
con = duckdb.connect(database=":memory:")

GOLD_SALES   = "gold/daily_sales_by_category.parquet"
GOLD_RETURNS = "gold/returns_rate.parquet"


# ============================================================
# Helper: run a check and print PASS / FAIL
#
#   rule_name    : human-readable label
#   fail_query   : SQL that returns ONLY the failing rows
#   table_path   : parquet file being checked
#
#   Logic:
#     - Execute the fail_query
#     - If zero rows come back  -> PASS
#     - If any rows come back   -> FAIL, show count + 3 examples
#
#   Returns True on PASS, False on FAIL.
# ============================================================
def run_check(rule_name, fail_query, table_path):
    fail_df = con.execute(fail_query).fetchdf()
    fail_count = len(fail_df)

    if fail_count == 0:
        print(f"  PASS  |  {rule_name}")
        return True
    else:
        print(f"  FAIL  |  {rule_name}")
        print(f"         |  Failing rows: {fail_count}")
        print(f"         |  Examples (up to 3):")
        sample = fail_df.head(3).to_string(index=False)
        for line in sample.split("\n"):
            print(f"         |    {line}")
        return False


# ============================================================
# CHECKS 1-4: daily_sales_by_category
# ============================================================

def check_1_sales_no_nulls():
    """No null order_date, region, or product_category."""
    return run_check(
        "Sales: no null order_date / region / product_category",
        f"""
        SELECT order_date, region, product_category,
               total_revenue, order_count
        FROM   '{GOLD_SALES}'
        WHERE  order_date        IS NULL
           OR  region            IS NULL
           OR  product_category  IS NULL
        """,
        GOLD_SALES,
    )


def check_2_revenue_positive():
    """total_revenue > 0 for all rows."""
    return run_check(
        "Sales: total_revenue > 0",
        f"""
        SELECT order_date, region, product_category,
               total_revenue
        FROM   '{GOLD_SALES}'
        WHERE  total_revenue <= 0
        """,
        GOLD_SALES,
    )


def check_3_order_count_positive():
    """order_count > 0 for all rows."""
    return run_check(
        "Sales: order_count > 0",
        f"""
        SELECT order_date, region, product_category,
               order_count
        FROM   '{GOLD_SALES}'
        WHERE  order_count <= 0
        """,
        GOLD_SALES,
    )


def check_4_sales_no_duplicate_grain():
    """No duplicate (order_date, region, product_category)."""
    return run_check(
        "Sales: unique grain (order_date, region, product_category)",
        f"""
        SELECT   order_date, region, product_category,
                 COUNT(*) AS row_count
        FROM     '{GOLD_SALES}'
        GROUP BY order_date, region, product_category
        HAVING   COUNT(*) > 1
        """,
        GOLD_SALES,
    )


# ============================================================
# CHECKS 5-8: returns_rate
# ============================================================

def check_5_returns_no_null_date():
    """No null order_date."""
    return run_check(
        "Returns: no null order_date",
        f"""
        SELECT *
        FROM   '{GOLD_RETURNS}'
        WHERE  order_date IS NULL
        """,
        GOLD_RETURNS,
    )


def check_6_rate_in_range():
    """returns_rate_pct between 0.0 and 100.0 inclusive."""
    return run_check(
        "Returns: returns_rate_pct in [0, 100]",
        f"""
        SELECT order_date, returned_orders, total_orders,
               returns_rate_pct
        FROM   '{GOLD_RETURNS}'
        WHERE  returns_rate_pct < 0.0
           OR  returns_rate_pct > 100.0
        """,
        GOLD_RETURNS,
    )


def check_7_returned_leq_total():
    """returned_orders <= total_orders."""
    return run_check(
        "Returns: returned_orders <= total_orders",
        f"""
        SELECT order_date, returned_orders, total_orders
        FROM   '{GOLD_RETURNS}'
        WHERE  returned_orders > total_orders
        """,
        GOLD_RETURNS,
    )


def check_8_date_range_span():
    """order_date range spans at least 30 days."""
    result = con.execute(f"""
        SELECT MIN(order_date) AS min_date,
               MAX(order_date) AS max_date,
               DATEDIFF('day', MIN(order_date), MAX(order_date)) AS span_days
        FROM   '{GOLD_RETURNS}'
    """).fetchone()

    min_date, max_date, span_days = result
    rule_name = "Returns: date range >= 30 days"

    if span_days >= 30:
        print(f"  PASS  |  {rule_name}  ({span_days} days: {min_date} to {max_date})")
        return True
    else:
        print(f"  FAIL  |  {rule_name}")
        print(f"         |  Span is only {span_days} days ({min_date} to {max_date})")
        return False


# ============================================================
# RUNNER: execute all 8 checks and print summary
# ============================================================
def run_all_checks():
    print("=" * 62)
    print("  GOLD LAYER DATA QUALITY CHECKS")
    print("=" * 62)

    checks = [
        check_1_sales_no_nulls,
        check_2_revenue_positive,
        check_3_order_count_positive,
        check_4_sales_no_duplicate_grain,
        check_5_returns_no_null_date,
        check_6_rate_in_range,
        check_7_returned_leq_total,
        check_8_date_range_span,
    ]

    print(f"\n  daily_sales_by_category ({GOLD_SALES})")
    print("-" * 62)
    results_sales = [checks[i]() for i in range(4)]

    print(f"\n  returns_rate ({GOLD_RETURNS})")
    print("-" * 62)
    results_returns = [checks[i]() for i in range(4, 8)]

    all_results = results_sales + results_returns
    passed = sum(all_results)
    total = len(all_results)

    print("\n" + "=" * 62)
    if passed == total:
        print(f"  RESULT: {passed}/{total} checks passed  --  ALL CLEAR")
    else:
        failed_names = []
        check_names = [
            "1-NullGrain", "2-Revenue>0", "3-OrderCount>0", "4-UniqueGrain",
            "5-NullDate", "6-RateRange", "7-Returned<=Total", "8-DateSpan",
        ]
        for name, result in zip(check_names, all_results):
            if not result:
                failed_names.append(name)
        print(f"  RESULT: {passed}/{total} checks passed")
        print(f"  FAILED: {', '.join(failed_names)}")
    print("=" * 62)

    return passed, total


# ============================================================
# Run when executed directly
# ============================================================
if __name__ == "__main__":
    run_all_checks()

  GOLD LAYER DATA QUALITY CHECKS

  daily_sales_by_category (gold/daily_sales_by_category.parquet)
--------------------------------------------------------------
  PASS  |  Sales: no null order_date / region / product_category
  FAIL  |  Sales: total_revenue > 0
         |  Failing rows: 7
         |  Examples (up to 3):
         |    order_date region product_category  total_revenue
         |    2024-03-01   East      Electronics        -400.64
         |    2024-04-27  South         Clothing        -114.78
         |    2024-04-29   West         Clothing         -49.63
  PASS  |  Sales: order_count > 0
  PASS  |  Sales: unique grain (order_date, region, product_category)

  returns_rate (gold/returns_rate.parquet)
--------------------------------------------------------------
  PASS  |  Returns: no null order_date
  PASS  |  Returns: returns_rate_pct in [0, 100]
  PASS  |  Returns: returned_orders <= total_orders
  PASS  |  Returns: date range >= 30 days  (363 days: 2024-01-03 to 20

# KATA 7.6

In [22]:
''' I have two gold tables: gold/daily_sales_by_category.parquet: order_date DATE, region VARCHAR, product_category VARCHAR, total_revenue FLOAT, order_count INTEGER gold/returns_rate.parquet: order_date DATE, total_orders INTEGER, returned_orders INTEGER, returns_rate_pct FLOAT 
Write a Streamlit app (app.py) that: 
Reads both parquet files with pandas
Title "Sales Performance Dashboard"
Sidebar date-range filter (default: last 30 days)
Chart 1: grouped bar — total_revenue by region (plotly express, color by product_category)
Chart 2: line — returns_rate_pct over time (plotly express)
Two metric cards: Total Revenue (sum), Average Returns Rate (mean)
"Data last updated: [max order_date]" at the bottom Read from local parquet via pd.read_parquet(), not hardcoded data. '''



' I have two gold tables: gold/daily_sales_by_category.parquet: order_date DATE, region VARCHAR, product_category VARCHAR, total_revenue FLOAT, order_count INTEGER gold/returns_rate.parquet: order_date DATE, total_orders INTEGER, returned_orders INTEGER, returns_rate_pct FLOAT \nWrite a Streamlit app (app.py) that: \nReads both parquet files with pandas\nTitle "Sales Performance Dashboard"\nSidebar date-range filter (default: last 30 days)\nChart 1: grouped bar — total_revenue by region (plotly express, color by product_category)\nChart 2: line — returns_rate_pct over time (plotly express)\nTwo metric cards: Total Revenue (sum), Average Returns Rate (mean)\n"Data last updated: [max order_date]" at the bottom Read from local parquet via pd.read_parquet(), not hardcoded data. '

In [24]:
! pip install streamlit pandas plotly pyarrow

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.9 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.9 MB 1.1 MB/s eta 0:00:09
   --- ------------------------------------ 0.8/9.9 MB 1.2 MB/s eta 0:00:08
   ---- ----------------------------------- 1.0/9.9 MB 1.2 MB/s eta 0:00:08
   ----- ---------------------------------- 1.3/9.9 MB 1.2 MB/s eta 0:00:08
   ------ --------------------------------- 1.6/9.9 MB 1.1 MB/s eta 0:00:08
   ------- -------------------------------- 1.8/9.9 MB 1.2 MB/s eta 0:00:07
   --------- ------------------------------ 2.4/9.9 MB 1.3 MB/s eta 0:00:06
   ---------- ----------------------------- 2.6/9.9 MB 1.3 MB/s eta 0:00:06
   ------------ --------------------------- 3.1/9.9 MB 1.4 MB/s eta 0:00:05
   ------------- --------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
! streamlit run app.py

'streamlit' is not recognized as an internal or external command,
operable program or batch file.
